# Jet-tagging with transformers
Notebook summarizing the most relevant parts of the openlab summer student project by Dino Cheng in 2026 at EP/SFT CERN, working on compressing a transformer-based jet tagger from heptokens with PQuantML using pruning via PDP/DST, quantization, and FITcompress.
It trains variants of a transformer-based jet tagger with various compression methods, both successful and unsuccessful.
A full report is available on Zenodo starting in (approx.) September 2026.
### Please do not execute this in the PQuantML repo! Move this into some other working directory due to import logics.

In [ ]:
%matplotlib inline

# pquant selects its backend from KERAS_BACKEND at import time (defaults to
# "tensorflow"). This project uses the torch layers (PQDense, etc.), so the
# variable must be set before any `pquant` import below.
import os

os.environ["KERAS_BACKEND"] = "torch"

# generic stuff
import logging
from abc import ABC, abstractmethod
from collections.abc import Callable
from datetime import datetime
from functools import partial
import logging
import json
from pathlib import Path
from typing import Literal

import joblib
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# torch
import torch
import torch as T
import torch.nn as nn
from torch.nn.functional import cross_entropy, gelu, relu
from torch.utils.data import DataLoader, Dataset, random_split

# lightning
import lightning as L
from lightning import LightningModule
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint
from torchmetrics import AUROC, Accuracy, ConfusionMatrix, F1Score

# onnx
import onnxruntime as ort

# PQuantML
from pquant.quantizer import Quantizer
from pquant import get_ebops, get_layer_keep_ratio
from pquant.core.hyperparameter_optimization import dst_config, pdp_config, wanda_config, fitcompress_config
from pquant.activations import PQActivation
from pquant.core.torch import convert_to_onnx
from pquant.core.torch.layers import (
    PQLayerNorm,
    call_post_round_functions,
    post_epoch_functions,
    post_pretrain_functions,
    pre_epoch_functions,
    pre_finetune_functions,
    save_weights_functions,
)
from pquant.core.torch.train import train_model
from pquant.core.torch.tracing import check_quantization
from pquant.layers import PQDense, PQMultiheadAttention, PQConv1d, PQConv2d, add_compression_layers, get_model_losses


# heptokens (Jeff's library, used for data loader functionality)
import heptokens
from heptokens.data.atlas_mappable import SingleFileMapModule
from heptokens.data.collation import preprocess_batch
from heptokens.models.token_classifier import (
    Embedder,
    Pooler,
    SequenceEncoder,
    ScheduledOptimiserMixin,
)



In [ ]:
def timestamp():
    return datetime.now().strftime("%m-%d-%H-%M-%S")

In [ ]:
def state_dict_export(model, path, verbose=True):
    sd = model.state_dict()
    if verbose:
        sd_serializable = {
            name: {
                "shape": list(tensor.shape),
                "dtype": str(tensor.dtype),
                "values": tensor.detach().cpu().numpy().tolist(),
            }
            for name, tensor in sd.items()
        }
        with open(path, "w") as f:
            json.dump(sd_serializable, f, indent=2)

    else:
        with open(path, "w") as f:
            for name, tensor in sd.items():
                t = tensor.detach().cpu().float()
                f.write(
                    f"{name}\tshape={list(t.shape)}\tdtype={tensor.dtype}\t"
                    f"min={t.min().item():.6f}\tmax={t.max().item():.6f}\t"
                    f"mean={t.mean().item():.6f}\n"
                )
                f.write(str(model.named_parameters))


In [ ]:
PREPROCESSING_ON = False # if True: requires preprocessing scalers from heptokens

In [ ]:
# torch configs
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.set_float32_matmul_precision("high")   # use TF32 on H100
torch.backends.cudnn.benchmark = True
torch.backends.cuda.enable_flash_sdp(True)

dtype_real = torch.float32

Some constants/names are defined in the following cell

In [ ]:
TRACK_FEATURES = [
    "pt",
    "deta",
    "dphi",
    "d0",
    "d0RelativeToBeamspot",
    "d0Uncertainty",
    "d0RelativeToBeamspotUncertainty",
    "z0RelativeToBeamspot",
    "z0RelativeToBeamspotUncertainty",
    "z0SinTheta",
    "z0SinThetaUncertainty",
    "lifetimeSignedD0",
    "lifetimeSignedD0Significance",
    "lifetimeSignedZ0SinTheta",
    "lifetimeSignedZ0SinThetaSignificance",
    "theta",
    "thetaUncertainty",
    "qOverP",
    "qOverPUncertainty",
    "ptfrac",
]

CLASS_NAMES = ["light", "c", "b", "tau"]

JET_FEATURES = ["pt", "mass", "eta", "phi"]

MAX_JET_PT = 7_000_000.0
MAX_CST_PT = 1_000_000.0
LABEL_KEY = "HadronConeExclTruthLabelID"
NUM_CSTS = 40

LABEL_MAP = {0: 0, 4: 1, 5: 2, 15: 3}  

# in the decision tree, they are ordered like 5 4 15 0
FLAVOUR_LABELS = {
    0: "light",
    4: "c",
    5: "b",
    15: "tau",
}

These are plotting functions from view_model.ipynb and view_run.ipynb.


In [ ]:
def collect_layer_bits(model):

    compressed_types = (PQConv1d, PQConv2d, PQDense)

    names = []
    integer_bits = []
    fractional_bits = []
    integer_bias_bits = []
    fractional_bias_bits = []

    for n, m in model.named_modules():
        if isinstance(m, compressed_types):
            k, i, f = m.get_weight_quantization_bits()
            names.append(n)
            integer_bits.append(float(np.mean(i.detach().cpu().numpy()) if torch.is_tensor(i) else i))
            fractional_bits.append(float(np.mean(f.detach().cpu().numpy()) if torch.is_tensor(f) else f))

            if hasattr(m, "get_bias_quantization_bits") and m.bias is not None:
                k_b, i_b, f_b = m.get_bias_quantization_bits()
                integer_bias_bits.append(float(np.mean(i_b.detach().cpu().numpy()) if torch.is_tensor(i_b) else i_b))
                fractional_bias_bits.append(float(np.mean(f_b.detach().cpu().numpy()) if torch.is_tensor(f_b) else f_b))
            else:
                integer_bias_bits.append(0.0)
                fractional_bias_bits.append(0.0)

    return names, integer_bits, fractional_bits, integer_bias_bits, fractional_bias_bits


def plot_bit_allocation(model, use_bias=False):
    names, integer_bits, fractional_bits, integer_bias_bits, fractional_bias_bits = collect_layer_bits(model)

    if not names:
        raise RuntimeError(
            "No compressed layers found. Make sure add_compression_layers() and "
            "FITCompress have been run on this model before plotting."
        )

    ibits = np.array(integer_bias_bits if use_bias else integer_bits, dtype=float)
    fbits = np.array(fractional_bias_bits if use_bias else fractional_bits, dtype=float)

    y = np.arange(len(names))
    fig_h = max(4, 0.28 * len(names))
    fig, ax = plt.subplots(figsize=(7, fig_h))

    ax.barh(y, -ibits, color="orange", label="Integer bits")
    ax.barh(y, fbits, color="blue", label="Fractional bits")

    ax.set_yticks(y)
    ax.set_yticklabels(names, fontsize=7)
    ax.invert_yaxis()
    ax.axvline(0, color="black", linewidth=0.8)

    max_bits = max(ibits.max(), fbits.max()) + 1
    ax.set_xlim(-max_bits, max_bits)
    ticks = ax.get_xticks()
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(abs(int(t))) for t in ticks])

    ax.set_xlabel("Bit-width")
    ax.set_title("Per-layer Bit Allocation (Integer vs Fractional) after FITCompress")
    ax.legend(loc="lower right")
    ax.grid(axis="x", linestyle="--", alpha=0.4)

    plt.tight_layout()
    return fig, ax

In [ ]:
def plot_remaining_weights(model):
    # Plot remaining weights
    names = []
    remaining = []
    total_w = []
    nonzeros = []

    integer_bits = []
    integer_bias_bits = []
    fractional_bits = []
    fractional_bias_bits = []

    for n, m in model.named_modules():
        if isinstance(m, (torch.nn.Conv1d, torch.nn.Conv2d, torch.nn.Linear, torch.nn.ReLU)):
            names.append(n.replace("encoder.transformer.encoder.layers.", "").replace("encoder.transformer.", ""))
            nonzero = np.count_nonzero(m.weight.detach().cpu())
            remaining_pct = nonzero / m.weight.numel()
            remaining.append(remaining_pct)
            total_w.append(m.weight.numel())
            nonzeros.append(nonzero)
            k, i, f = m.get_weight_quantization_bits()
            k_b, i_b, f_b = m.get_bias_quantization_bits()

            integer_bits.append(i)
            integer_bias_bits.append(i)
            fractional_bias_bits.append(f)

    #nonzeros = np.ndarray(nonzeros)
    #total_w = np.ndarray(total_w)

    total_remaining_ratio = sum(nonzeros) / sum(total_w)

    total_n_weights = sum(p.numel() for p in model.parameters() if p.requires_grad)

    fig1, ax = plt.subplots(1, 2, figsize=(10, 6))

    ax[0].barh(range(len(names)), remaining)
    ax[0].set_yticks(range(len(names)))
    ax[0].set_yticklabels(names)
    ax[0].invert_yaxis()  

    new_xtick = []
    for i in ax[0].get_xticklabels():
        xtick = f"{float(i.get_text()) * 100}%"
        new_xtick.append(xtick)
    ax[0].set_xticklabels(new_xtick)
    ax[0].title.set_text(f"Remaining weights per layer (%) \n Total ratio of remaining weights: {total_remaining_ratio*100:.2f} %")

    ax[1].barh(range(len(nonzeros)), total_w, color="lightcoral", label="pruned weights")
    ax[1].barh(range(len(nonzeros)), nonzeros, color="steelblue", label="nonzero weights")
    ax[1].set_yticks(range(len(names)))
    ax[1].set_yticklabels(names)
    ax[1].invert_yaxis()  # keep first layer at the top
    ax[1].title.set_text(f"Weights per layer \n Total number of weights: {total_n_weights} ")
    ax[1].legend()

    plt.tight_layout()
    return fig1, ax


# this will be left unused in this notebook.
def plot_model_ebops(model):
    fig2, ax2_1 = plt.subplots(figsize=(7, 5))
    ax2_2 = ax2_1.twiny()

    # fill the metrics yourself!
    reference_ebops = 8.9e9
    pruned_ebops = get_ebops(model)
    pruned_quantized_smaller_architecture_ebops = 2.02e8
    compressed_smaller_architecture_ebops_2 = 3e7
    fpga_ebops = 350_000 # doi.org/10.22323/1.485.0081


    labels = ["Uncompressed", "Pruned \n (DST)", "Pruned \n (DST) \n & \n Quantized \n (10-bit data, \n 6-bit weights)"]
    values = [reference_ebops, pruned_ebops, pruned_quantized_smaller_architecture_ebops]
    accuracys = [71.6, 68.7, 65.2]
    x = np.arange(len(labels))
    width = 0.5

    ax2_2.set_xscale("log")
    ax2_2.set_ylabel("Models")
    ax2_2.set_xlabel("EBOPs (log.)")

    h = 0.35
    ax2_1.barh(x + h/2, accuracys, height=h, color="red", label="Accuracy")
    ax2_2.barh(x - h/2, values, height=h, color="blue", label="EBOPs")

    
    ax2_1.set_xlabel("Accuracy (%)")
    ax2_1.set_xlim(0, 100)
    ax2_2.axvline(fpga_ebops, linestyle='dashed', color="navy", label='EBOPs on FPGAs')

    handles1, labels1 = ax2_2.get_legend_handles_labels()
    handles2, labels2 = ax2_1.get_legend_handles_labels()
    ax2_2.legend(handles1 + handles2, labels1 + labels2, loc="best")

    ax2_2.set_yticks(x)
    ax2_2.set_yticklabels(labels, rotation=0, va="center")
    ax2_2.invert_yaxis()  # "Reference" on top, matching original left-to-right order

    plt.suptitle(f"EBOPs comparison of compressed classifier")

    plt.tight_layout()
    return fig2, (ax2_1, ax2_2)



The original model is trained on the JetSet dataset: [mc-flavtag-ttbar-small.h5](https://opendata.cern.ch/record/93940) , in which the "light" class is overrepresents. Hence, it is resampled into a 100k- and 1000k-sample dataset with equal amount of samples per class, which we load into the heptoken.data.SingleFileMapModule class. If you need to resample yourself, use view_data.ipynb.

In [ ]:
data_path = Path("/shared/data/ttbar-1000k_balanced.h5")

In [ ]:
# loading 1M samples
n_points = 1_000_000
dataset = SingleFileMapModule(
        data_path=data_path,
        batch_size=1024,
        n_classes=4,
        num_workers=1,
        persistent_workers=True,
        jet_features=JET_FEATURES,
        cst_features=TRACK_FEATURES,
        max_jet_pt=7_000_000,
        max_cst_pt=1_000_000,
        num_jets=n_points,
        num_csts=n_points
        )


This is a LLM-inspired function that analyzes the dataset for us.

In [ ]:
def mapdataset_stats(split):
    """
    Compute track-feature statistics and label counts for a split returned by
    SingleFileMapModule: dm.train_set, dm.valid_set, or dm.test_set.
    """
    ds = split.dataset          # underlying MapDataset
    idx = np.asarray(split.indices)

    csts = ds.data_dict["csts"][idx]       # [n_jets, n_csts, n_features]
    mask = ds.data_dict["mask"][idx].astype(bool)  # [n_jets, n_csts]
    labels = ds.data_dict["labels"][idx]

    # Select only valid constituents, flatten to [n_valid_tracks, n_features].
    valid_csts = csts[mask]

    if valid_csts.shape[0] == 0:
        raise ValueError("This split contains no valid constituents.")

    mean = valid_csts.mean(axis=0)
    std = valid_csts.std(axis=0)

    # rowvar=False: features are variables/columns
    cov = np.cov(valid_csts, rowvar=False)
    corrcoef = np.corrcoef(valid_csts, rowvar=False)
    return {
        "n_jets": len(idx),
        "n_tracks": int(mask.sum()),
        "mean": mean,
        "std": std,
        "cov": cov,
        "corr": corrcoef,
        "bincount": np.bincount(labels, minlength=4),
        "labels": labels,
    }

In [ ]:
dataset.setup("fit")  # harmless for SingleFileMapModule; splits already exist

train_stats = mapdataset_stats(dataset.train_set)
val_stats = mapdataset_stats(dataset.valid_set)
test_stats = mapdataset_stats(dataset.test_set)

This computes the metrics and plot them in the next cell. We see the magnitude of each feature (=input dimension), the distribution of the classes and the correlation matrix.

In [ ]:
for name, s in [("TRAIN", train_stats), ("VAL", val_stats)]:
    print(f"\n=== {name} ===")
    print("n_jets:", s["n_jets"], " n_valid_tracks:", s["n_tracks"])
    for feat, m, sd in zip(TRACK_FEATURES, s["mean"], s["std"]):
        print(f"{feat:45s} mean={m:.4f}  std={sd:.4f}")
    print("corr shape:", s["corr"].shape)
    print("bincount (light, c, b, tau):", s["bincount"].tolist())

In [ ]:
x = np.arange(len(TRACK_FEATURES))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width/2, train_stats["mean"], width, yerr=train_stats["std"], label="Train", capsize=3)
ax.bar(x + width/2, val_stats["mean"], width, yerr=val_stats["std"], label="Val", capsize=3)
ax.set_xticks(x)
ax.set_xticklabels(TRACK_FEATURES, rotation=60, ha="right")
ax.set_ylabel("Value")
ax.set_title("Feature mean/std: train vs validation")
ax.legend()
plt.tight_layout()

plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
xc = np.arange(len(CLASS_NAMES))
ax.bar(xc - width/2, train_stats["bincount"], width, label="Train")
ax.bar(xc + width/2, val_stats["bincount"], width, label="Val")
ax.set_yscale("log")
ax.set_xticks(xc)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel("Jet count (log)")
ax.set_title("Jet flavor class counts: train vs validation")
ax.legend()
plt.tight_layout()

plt.show()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(train_stats["corr"], cmap="RdBu",
                vmin=-np.abs(train_stats["corr"]).max(), vmax=np.abs(train_stats["corr"]).max())
ax.set_xticks(np.arange(len(TRACK_FEATURES)))
ax.set_yticks(np.arange(len(TRACK_FEATURES)))
ax.set_xticklabels(TRACK_FEATURES, rotation=60, ha="right", fontsize=8)
ax.set_yticklabels(TRACK_FEATURES, fontsize=8)
ax.set_title("Correlation matrix (train split)")
fig.colorbar(im, ax=ax)
plt.tight_layout()

plt.show()

We now construct the machine learning model from feature_clf_model, using the class FeatureClassifier. The layers in token_classifier.py and transformer.py are exchanged from torch.nn layers to PQuantML layers.
The files are pasted in the next two cells.

This is essentially the architecture we use:

![Architecture](/shared/figs/arch.png)

In [ ]:
"""transformer.py
Transformer building block for sequence-aware encoding.

Extracted verbatim from heptokens.models.transformer (only the ``Transformer``
class is needed by the FeatureClassifier).
"""

def make_data_quantizer(config) -> Quantizer:
    """Build a per-tensor data-lane quantizer from a pquant config.

    Matches how pquant's automatic pass constructs data-edge quantizers
    (see pquant.core.torch.tracing._insert_missing_quantizers), so manually
    inserted activation quantizers are consistent with the auto-inserted ones.
    """
    qp = config.quantization_parameters
    return Quantizer(
        k=qp.default_data_keep_negatives,
        i=qp.default_data_integer_bits,
        f=qp.default_data_fractional_bits,
        overflow=qp.overflow_mode_data,
        round_mode=qp.round_mode,
        is_heterogeneous=False,
        is_data=True,
        granularity="per_tensor",
        hgq_gamma=qp.hgq_gamma,
    )

class PQTransformerEncoder(nn.Module):
    """
    PQTransformerEncoder
    """

    def __init__(
        self,
        config,
        num_layers: int,
        d_model: int,
        nhead: int,
        dim_feedforward: int = 2048,
        activation="relu",
        layer_norm_eps: float = 1e-5,
        norm_first: bool = False,
        bias: bool = True,
        norm: nn.Module | None = None,
    ) -> None:
        super().__init__()
        self.layers = nn.ModuleList(
            [
                PQTransformerEncoderLayer(
                    config,
                    d_model=d_model,
                    nhead=nhead,
                    dim_feedforward=dim_feedforward,
                    activation=activation,
                    layer_norm_eps=layer_norm_eps,
                    norm_first=norm_first,
                    bias=bias,
                )
                for _ in range(num_layers)
            ]
        )
        self.num_layers = num_layers
        self.norm = norm

    def forward(
        self,
        src: torch.Tensor,
        mask: torch.Tensor | None = None,
        src_key_padding_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        output = src
        for mod in self.layers:
            output = mod(
                output,
                src_mask=mask,
                src_key_padding_mask=src_key_padding_mask,
            )
            #print(f"TransformerEncoder, output {output}")
        if self.norm is not None:
            output = self.norm(output)
        return output
    

class PQTransformerEncoderLayer(nn.Module):
    r"""
    PQ'd version of torch.nn.TransformerEncoderLayer
    TransformerEncoderLayer is made up of self-attn and feedforward network.

    """

    __constants__ = ["norm_first"]

    def __init__(
        self,
        config,
        d_model: int,
        nhead: int,
        dim_feedforward: int = 2048,
        activation: str | Callable[[torch.Tensor], torch.Tensor] = "relu",
        layer_norm_eps: float = 1e-5,
        batch_first: bool = False,
        norm_first: bool = False,
        bias: bool = True,
    ) -> None:
        super().__init__()
        self.self_attn = PQMultiheadAttention(
            config,
            embed_dim=d_model,
            num_heads=nhead,
            bias=bias,
            batch_first=True,
            quantize_input=True,
            quantize_output=True,
        )

        self.input_quantizer = make_data_quantizer(config)

        # Feed-forward network.
        self.linear1 = PQDense(config, d_model, dim_feedforward, bias=bias, quantize_output=False) # false because it goes into self.activation either way
        self.linear2 = PQDense(config, dim_feedforward, d_model, bias=bias, quantize_output=True)

        self.norm_first = norm_first
        self.norm1 = PQLayerNorm(config, d_model, eps=layer_norm_eps, bias=bias, quantize_output=True) # input is quantized
        self.norm2 = PQLayerNorm(config, d_model, eps=layer_norm_eps, bias=bias, quantize_output=False) 

        self.activation = PQActivation(config, activation, quantize_input=True) # input from linear1
        

    def forward(
        self,
        src: torch.Tensor,
        src_mask: torch.Tensor | None = None,
        src_key_padding_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        x = self.input_quantizer(src)
        # print(f"TransformerEncoderLayer pre {x}")
        if self.norm_first:
            x = x + self._sa_block(self.norm1(x), src_mask, src_key_padding_mask)
            x = x + self._ff_block(self.norm2(x))
        # default
        else:
            x = self.norm1(x + self._sa_block(x, src_mask, src_key_padding_mask))
            x = self.norm2(x + self._ff_block(x))

        # print(f"TransformerEncoderLayer post sa/ff/norm {x}")
        return x

    # self-attention block
    def _sa_block(
        self,
        x: torch.Tensor,
        attn_mask: torch.Tensor | None,
        key_padding_mask: torch.Tensor | None,
    ) -> torch.Tensor:
        x = self.self_attn(
            x,
            x,
            x,
            key_padding_mask=key_padding_mask,
            attn_mask=attn_mask,
            need_weights=False
        )[0]
        #print(f"Post Self Attention {x}")
        return x

    # feed-forward block
    def _ff_block(self, x: torch.Tensor) -> torch.Tensor:
        #print(f"Pre-FF (lin(act(lin(x)))) {x}")
        return self.linear2(self.activation(self.linear1(x))) 
    

class Transformer(nn.Module):
    """
    Transformer encoder stack with input/output projections.

    Preserves the input sequence length and supports a padding mask.
    """

    def __init__(
        self,
        config,
        input_dim: int,
        output_dim: int,
        *,
        d_model: int = 128,
        n_heads: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 512,
        activation = "relu",
    ) -> None:
        super().__init__()
        self.input_proj = PQDense(config, input_dim, d_model)
        """
        layer = PQTransformerEncoderLayer(
            config,
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            activation=activation,
            batch_first=True,
        )"""
        self.encoder = PQTransformerEncoder(
            config, 
            num_layers=num_layers, 
            d_model=d_model, 
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            activation=activation
            )
        self.output_proj = PQDense(config, d_model, output_dim, quantize_output=True)

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        """Forward pass through the transformer.

        Args:
            x: Tensor of shape [batch, n_csts, input_dim].
            mask: Boolean tensor of shape [batch, n_csts] where True means valid.

        Returns:
            Tensor of shape [batch, n_csts, output_dim].
        """
        squeeze_batch = False
        # removed for PQ
        """
        if x.dim() == 2:
            x = x.unsqueeze(0)
            squeeze_batch = True
        """

        key_padding_mask = None
        if mask is not None:
            # mask = torch.tensor(mask, dtype=torch.bool)# removed for ONNX conversion, assuming mask is boolean tensor
            # removed for PQ
            """
            if mask.dim() == 1:
                mask = mask.unsqueeze(0)
            """
            # Zero out padded positions to avoid NaNs in attention/FFN paths.
            
            x *= mask.to(torch.float32).unsqueeze(-1)
            key_padding_mask = ~mask

            # Remove for PQ
            """
            if empty_sequences.any():
                mask_for_encoder = mask.clone()
                # Ensure at least one valid token so attention doesn't see all padding.
                mask_for_encoder[empty_sequences, 0] = True
                key_padding_mask = ~mask_for_encoder
                x[empty_sequences] = 0.0
            else:
                key_padding_mask = ~mask
            """
        x = self.input_proj(x)
        #print("Transformer input proj")
        #print(x)
        x = self.encoder(x, src_key_padding_mask=key_padding_mask)
        #print(f"Transformer encoder {x} with key padding mask {key_padding_mask}")

        x = self.output_proj(x)

        if mask is not None:
            x *= mask.to(torch.float32).unsqueeze(-1)
            # Remove for PQ
            """
            if empty_sequences is not None and empty_sequences.any():
                x[empty_sequences] = 0.0
            """

        if squeeze_batch:
            x = x.squeeze(0)

        #print(f"Output projection {x}")
        return x


In [ ]:
"""token_classifier.py
Standalone FeatureClassifier (and its base JetClassifier).

Extracted from heptokens.models.token_classifier. Only the classes reachable
from ``FeatureClassifier`` are kept:

    FeatureClassifier -> JetClassifier -> {FeatureEmbedder, TransformerEncoder,
    MeanPooler / MaxPooler / ClsTokenPooler}

The VQ-VAE-dependent embedders (TokenEmbedder, VectorEmbedder) and classifiers
(TokenClassifier, VectorClassifier) are intentionally omitted -- they require a
tokenizer checkpoint and are not part of this handoff.

Architecture (FeatureClassifier):
    {"csts": (B, N, 20), "mask": (B, N) bool}
      -> FeatureEmbedder (PQDense 20 -> d_model)
      -> TransformerEncoder (Transformer encoder stack, mean/max/cls)
      -> Pooler -> PQDense(d_model -> n_classes) -> logits (B, n_classes)
"""


class TransformerEncoder(SequenceEncoder):
    """Transformer-based sequence encoder."""

    def __init__(
        self,
        config,
        input_dim: int,
        d_model: int,
        n_heads: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 512,
        activation = "relu",
    ) -> None:
        super().__init__()
        self.d_model = d_model
        self.transformer = Transformer(
            config,
            input_dim=d_model,
            output_dim=d_model,
            d_model=d_model,
            n_heads=n_heads,
            num_layers=num_layers,
            dim_feedforward=dim_feedforward,
            activation=activation,
        )

    @property
    def output_dim(self) -> int:
        return self.d_model

    def encode(self, x: T.Tensor, mask: T.BoolTensor) -> T.Tensor:
        return self.transformer(x, mask=mask)


class MeanPooler(Pooler):
    """Mean pooling over valid positions."""

    def __init__(self, config) -> None:
        super().__init__()
        # Quantize the genuine data operands of the masked mean: the input
        # activation, the accumulated sum, and the per-jet count (the divisor).
        # The mask itself is a structural {0, 1} selector, not a data activation,
        # so it is left unquantized.
        self.x_quantizer = make_data_quantizer(config)
        self.sum_quantizer = make_data_quantizer(config)
        self.count_quantizer = make_data_quantizer(config)
        self.num_csts = config.training_parameters.num_csts

    def pool(self, x: T.Tensor, mask: T.BoolTensor) -> T.Tensor:
        x = self.x_quantizer(x) 
        mask_f = mask.to(T.float32)
        valid_sum = self.sum_quantizer((x * mask_f.unsqueeze(-1)).sum(dim=1))        
        valid_count = mask_f.sum(dim=1, keepdim=True).clamp(min=1) # no quantizer because mask is 0/1
        
        valid_ratio = self.count_quantizer(valid_sum / valid_count) # quantizer because this is a ratio
        return valid_ratio
    
class MaxPooler(Pooler):
    """Max pooling over valid positions."""

    def pool(self, x: T.Tensor, mask: T.BoolTensor) -> T.Tensor:
        x_masked = x.masked_fill(~mask.unsqueeze(-1), float("-inf"))
        return x_masked.max(dim=1)[0]

class FeatureEmbedder(Embedder):
    """Embedder for raw constituent features."""

    def __init__(self, config, input_dim: int, d_model: int) -> None:
        super().__init__()
        self.input_dim = input_dim
        self.d_model = d_model
        self.projection = PQDense(config, input_dim, d_model, quantize_input=True)

    @property
    def output_dim(self) -> int:
        return self.d_model

    def embed(self, csts, mask) -> T.Tensor:
        csts = T.nan_to_num(csts, nan=0.0)
        # Zero out masked positions
        mask_f = mask.to(T.float32)
        mask_expanded = mask_f.unsqueeze(-1)
        csts = csts * mask_expanded
        
        #csts = T.where(mask.unsqueeze(-1), csts, T.zeros_like(csts))

        # embeddings = self.projection(csts) put directly in mask argument
        return self.projection(csts)


class JetClassifier(ScheduledOptimiserMixin, LightningModule):
    """General-purpose jet classifier with pluggable components.

    Architecture:
        Input -> Embedder -> Encoder -> Pooler -> Head -> Logits
    """

    def __init__(
        self,
        *,
        config,
        embedder: Embedder,
        encoder: SequenceEncoder,
        pooler: Pooler,
        n_classes: int,
        learning_rate: float = 1e-3,
        **kwargs,
    ) -> None:
        super().__init__()
        self.save_hyperparameters(ignore=["embedder", "encoder", "pooler", "layers"])

        self.F = config.training_parameters.num_features # 20
        self.N = config.training_parameters.num_csts # 40

        self.input_shape = (self.N, self.F+1)

        # remove this because this is only valid for original implementation, but we replace it with a PQ'd version
        """# Validate compatibility
        if embedder.output_dim != encoder.encode.__code__.co_varnames[1:2][0]:  # Quick check
            log.warning(
                f"Embedder output_dim ({embedder.output_dim}) may not match "
                f"encoder input_dim. Check compatibility."
            )"""

        self.embedder = embedder
        self.encoder = encoder
        self.pooler = pooler
        self.n_classes = n_classes

        self.config = config
        self.training_config = self.config.training_parameters

        # Classification head
        self.classifier = PQDense(config, encoder.output_dim, n_classes, quantize_output=True)

        # Metrics
        self.train_acc = Accuracy("multiclass", num_classes=n_classes, average="macro")
        self.valid_acc = Accuracy("multiclass", num_classes=n_classes, average="macro")
        self.test_acc = Accuracy("multiclass", num_classes=n_classes, average="macro")

        self.train_acc_classwise = Accuracy("multiclass", num_classes=n_classes, average="none")
        self.valid_acc_classwise = Accuracy("multiclass", num_classes=n_classes, average="none")
        self.test_acc_classwise = Accuracy("multiclass", num_classes=n_classes, average="none")

        # AUC metrics (one-vs-rest for multiclass)
        self.train_auc = AUROC(task="multiclass", num_classes=n_classes, average="macro")
        self.valid_auc = AUROC(task="multiclass", num_classes=n_classes, average="macro")
        self.test_auc = AUROC(task="multiclass", num_classes=n_classes, average="macro")

        
        # F1 metrics (one-vs-rest for multiclass)
        self.train_f1 = F1Score(task="multiclass", num_classes=n_classes, average="macro")
        self.valid_f1 = F1Score(task="multiclass", num_classes=n_classes, average="macro")
        self.test_f1 = F1Score(task="multiclass", num_classes=n_classes, average="macro")

        self.valid_confusion_matrix = ConfusionMatrix(task="multiclass", num_classes=n_classes, normalize="true")
        self.test_confusion_matrix = ConfusionMatrix(task="multiclass", num_classes=n_classes, normalize="true")

        self.class_weights = T.ones(n_classes)

        # Store outputs for ROC plotting
        self.validation_outputs = []

    def forward(self, input: T.Tensor) -> T.Tensor:
        """Single-tensor version instead of dict version"""
        csts = input[:, :, :self.F]
        mask = input[:, :, self.F] > 0 # implicit conversion back from csts.dtype to bool

        x = self.embedder.embed(csts, mask)
        x = self.encoder.encode(x, mask)
        x = self.pooler.pool(x, mask)
        return self.classifier(x)
    
    def pq_loss(self, output, labels, loss_fn=cross_entropy):
        loss = loss_fn(output, labels, label_smoothing=0.01).to(output.device)
        compression_loss = get_model_losses(self, T.tensor(0.).to(output.device))
        return loss + compression_loss

    def _shared_step(self, batch: dict, prefix: str) -> T.Tensor:
        x, labels = batch
        x = x.to(device=self.device)
        labels = labels.to(device=self.device)

        output = self.forward(x)

        loss = self.pq_loss(output, labels) 

        self.log(f"{prefix}/total_loss", loss)

        acc = getattr(self, f"{prefix}_acc")
        acc(output, labels)
        self.log(f"{prefix}/acc", acc)

        acc_classwise = getattr(self, f"{prefix}_acc_classwise")
        acc_per_class = acc_classwise(output, labels)
        for i, acc in enumerate(acc_per_class):
            self.log(f"{prefix}/acc_class_{i}_{CLASS_NAMES[i]}", acc, on_epoch=True)

        auc = getattr(self, f"{prefix}_auc")
        probs = T.softmax(output, dim=1)
        auc(probs, labels)
        self.log(f"{prefix}/auc", auc)

        f1 = getattr(self, f"{prefix}_f1")
        f1(probs, labels)
        self.log(f"{prefix}/f1", f1)

        if prefix == "test":
            self.test_confusion_matrix(probs, labels)
        elif prefix == "valid":
            self.valid_confusion_matrix(probs, labels)
        return loss

    def training_step(self, batch_dict: dict) -> T.Tensor:
        return self._shared_step(batch_dict, "train")

    def validation_step(self, batch_dict: dict) -> T.Tensor:
        return self._shared_step(batch_dict, "valid")

    def predict_step(self, batch: dict) -> dict:
        x, labels = batch

        output = self.forward(x)

        return {"output": output, "label": labels.unsqueeze(-1)}
    
    def test_step(self, batch: dict) -> dict:
        return self._shared_step(batch, "test")
    
    def on_train_epoch_start(self):
        if self.current_epoch == 0:
            pass # initial
        self.train()
        pre_epoch_functions(self, self.current_epoch, self.training_config.pretraining_epochs)
        if self.training_config.rounds == 0:
            if (self.current_epoch == self.training_config.pretraining_epochs + self.training_config.epochs - 1) and (self.training_config.fine_tuning_epochs != 0):
                print("Finetuning starting")
                pre_finetune_functions(self)
        else:
            if (self.current_epoch ==  self.training_config.pretraining_epochs + self.training_config.rounds * self.training_config.epochs - 1) and (self.training_config.fine_tuning_epochs != 0):
                print("Finetuning starting")
                pre_finetune_functions(self)
    
    def on_train_epoch_end(self):
        # general post epoch function
        post_epoch_functions(self, self.current_epoch, self.training_config.pretraining_epochs)
    
    def on_validation_epoch_start(self):
        self.eval()
        
    def on_validation_epoch_end(self):
        self.to(self.device)
        self.log(f"valid/remaining_weights", get_layer_keep_ratio(self))
        self.log(f"valid/EBOPs", get_ebops(self))

        conf_matrix_plot, conf_matrix_ax = self.valid_confusion_matrix.plot(
            labels=CLASS_NAMES
        )
        conf_matrix_plot.set_dpi(600)

        if self.logger is not None and hasattr(self.logger.experiment, "add_figure"):
            self.logger.experiment.add_figure("valid/confusion_matrix", conf_matrix_plot, self.current_epoch)
        else:
            print("No logger configured. Image is deleted")
    
        plt.close(conf_matrix_plot)

        self.valid_confusion_matrix.reset()

        # post pretrain
        if (self.current_epoch == self.training_config.pretraining_epochs - 1) or (self.training_config.pretraining_epochs == 0 and self.current_epoch == 0):
            try:
                print(f"{self.trainer.train_dataloader=}")
            except Exception as e:
                print(e)
            train_dl = self.trainer.train_dataloader if hasattr(self, 'trainer') else None
            with T.enable_grad():
                post_pretrain_functions(
                self, self.config, input_shape=self.input_shape, train_loader=train_dl, loss_function=self.pq_loss
                ) # after validation
            print("Pretraining ended")

    def on_test_epoch_end(self):
        conf_matrix_plot, conf_matrix_ax = self.test_confusion_matrix.plot(
            labels=CLASS_NAMES
        )
        conf_matrix_plot.tight_layout()

        if self.logger is not None and hasattr(self.logger.experiment, "add_figure"):
            self.logger.experiment.add_figure("valid/confusion_matrix", conf_matrix_plot, self.current_epoch)
        else:
            print("No logger configured. Image is deleted")
    
        plt.close(conf_matrix_plot)

        self.test_confusion_matrix.reset()


class FeatureClassifier(JetClassifier):
    """Explicit instantiation of JetClassifier using raw feature embedder."""

    def __init__(
        self,
        *,
        config,
        data_sample: dict | None = None,
        n_classes: int,
        d_model: int = 128,
        n_heads: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 512,
        activation: str = "relu",
        pooling: Literal["mean", "max", "cls"] = "mean",
        learning_rate: float = 1e-3,
        input_dim: int | None = None,
        **kwargs,
    ) -> None:
        # The original model reads the input feature dim from a data sample.
        # We also accept an explicit ``input_dim`` so the model can be built
        # without a data sample (the checkpoint does store ``data_sample``).
        if data_sample is not None:
            input_dim = data_sample["csts"].shape[-1]
        if input_dim is None:
            raise ValueError(
                "Provide either `data_sample` (dict with 'csts') or `input_dim`."
            )

        embedder = FeatureEmbedder(
            config,
            input_dim=input_dim,
            d_model=d_model,
        )
        encoder = TransformerEncoder(
            config,
            input_dim,
            d_model=d_model,
            n_heads=n_heads,
            num_layers=num_layers,
            dim_feedforward=dim_feedforward,
            activation=activation,
        )
        pooler_cls = {
            "mean": MeanPooler,
            "max": MaxPooler,
        }[pooling]
        pooler = pooler_cls(d_model, config) if pooling == "cls" else pooler_cls(config)

        super().__init__(
            config=config,
            embedder=embedder,
            encoder=encoder,
            pooler=pooler,
            n_classes=n_classes,
            learning_rate=learning_rate,
            **kwargs,
        )

def warmup_cosine_scheduler(
    optimizer,
    warmup_epochs: int = 5,
    min_lr: float = 1e-6,
    model=None,
    max_epochs: int = -1,
):
    """Linear warmup followed by cosine annealing, configured in epochs.

    If ``max_epochs`` is -1 (default), it is read from ``model.trainer``.
    """
    if max_epochs < 1 and model is not None:
        max_epochs = model.trainer.max_epochs
    if max_epochs < 1:
        raise ValueError("max_epochs must be positive (set it or pass a model with a trainer).")

    warmup = min(warmup_epochs, max_epochs)
    warmup_sched = T.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=1e-2, total_iters=warmup,
    )
    cosine_sched = T.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(max_epochs - warmup, 1), eta_min=min_lr,
    )
    return T.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_sched, cosine_sched],
        milestones=[warmup],
    )



In [ ]:
time_str = timestamp()
save_path = f"/shared/logs/{time_str}"
onnx_path = f"{save_path}/model_last.onnx"

os.makedirs(save_path, exist_ok=True)

logging.basicConfig(
level=logging.INFO,
format="%(asctime)s [%(levelname)s] %(message)s",
handlers=[logging.FileHandler(f"{save_path}/output.log", mode='w+'), logging.StreamHandler()],
)


This loads preprocessing scripts from heptokens, namely normalizers

In [ ]:
if PREPROCESSING_ON:
    jet_q_path = Path("/shared/pqjetset_standalone/checkpoint/jet_quantiles.joblib")
    cst_q_path = Path("/shared/pqjetset_standalone/checkpoint/cst_quantiles.joblib")
    jet_q = joblib.load(jet_q_path)
    cst_q = joblib.load(cst_q_path)
    preprocess = partial(preprocess_batch, cst_fn=cst_q, jet_fn=jet_q)
else:
    preprocess = lambda x: x # do nothing

We load a standard config from PQuantML for the respective pruning algorithm. We start with PDP, and adapt the preloaded values wherever required. Test out different sparsity values and unstructured/structured flags! You will notice that structured pruning only allows a fraction of the sparsity possibl with unstructured pruning.

The training should take approx. 4 hours, but it is possible to just decrease the amount of epochs to something like 10/10/5 instead of 15/30/10.

In [ ]:
config = pdp_config() 

config.training_parameters.pretraining_epochs = 15 # training before start of pruning
config.training_parameters.epochs = 30 # training with pruning
config.training_parameters.fine_tuning_epochs = 10 # training after end of pruning, pruning mask is fixed here
max_epochs = config.training_parameters.pretraining_epochs + config.training_parameters.epochs + config.training_parameters.fine_tuning_epochs
# added manually to config for tuple input of model
config.training_parameters.num_csts = 40 # amount of constituents
config.training_parameters.num_features = 20 # amount of features

config.quantization_parameters.enable_quantization = True
config.quantization_parameters.granularity = "per_weight"
# granularity -> per tensor/channel/weight 
config.quantization_parameters.use_relu_multiplier = False
config.quantization_parameters.use_high_granularity_quantization = False
config.quantization_parameters.hgq_beta = 1e-13
config.quantization_parameters.hgq_gamma = 1e-6
config.quantization_parameters.overflow_mode_parameters = "SAT"
config.quantization_parameters.overflow_mode_data = "SAT"

config.quantization_parameters.default_data_keep_negatives = 0.
config.quantization_parameters.dynamic_data_quantization = True
config.quantization_parameters.default_data_integer_bits = 0.
config.quantization_parameters.default_data_fractional_bits = 8. # give all weights to fractional by default if dynamical data quantization

config.quantization_parameters.default_weight_keep_negatives = 1.
config.quantization_parameters.default_weight_integer_bits = 0.
config.quantization_parameters.default_weight_fractional_bits = 6. # give all weights to fractional by default if dynamical data quantization

config.pruning_parameters.enable_pruning = True
#config.pruning_parameters.max_pruning_pct = 1.00 # DST setting
#config.pruning_parameters.alpha = 1e-3 # DST setting
config.pruning_parameters.epsilon = 0.037
config.pruning_parameters.sparsity = 0.90
config.pruning_parameters.structured_pruning = False

config.training_parameters.num_csts = 40   
config.training_parameters.num_features = 20    

The original model has a (d_model, dim_feedforward, num_layers) of (128, 512, 4), which is strongly overdimensioned. 
Reducing it to (128, 128, 4) is possible without hurting performance, but accelerates training.
Reducing it until (64, 64, 3) is possible with some performance loss (f1 = 0.6, f1_original = 0.66)
Model for Sanjiban is trained with (1024, 1024, 4).
All information should also be stored in the .yaml file created by Lightning.

In [ ]:
n_classes = 4
d_model = 128
n_heads = 8 # does not change EBOPs!
num_layers = 4
dim_feedforward = 512
activation = 'relu' # originally gelu, but for compression, usually RELU is used because its easier
learning_rate = 1e-3
pooling = 'mean'
input_dim = config.training_parameters.num_features 

model = FeatureClassifier(
    config=config, 
    n_classes=n_classes, 
    d_model=d_model,
    n_heads=n_heads,
    num_layers=num_layers,
    dim_feedforward=dim_feedforward,
    activation=activation,
    learning_rate=learning_rate,
    pooling=pooling,
    # these are tunable hyperparameters as well, but they are also pruning/quantization-dependent and do not have that big influence compared to learning rate etc.
    scheduler=partial(warmup_cosine_scheduler, warmup_epochs=3, min_lr=1e-5),
    input_dim=input_dim
    )
add_compression_layers(model, config)

This searches missing quantizers and adds them automatically. 
Theoretically, one could also define a pure torch/keras model and run this function to add the compressors.

We move the model entirely to device (="cuda").
In the hpo loop, this needs to be called in each iteration, as otherwise, the new instance of the model would be created on cpu. Especially quantizers are prone to be reinitialized on cpu despite having been on cuda previously.

We make an initializing forward pass with a random input. 

In [ ]:
# Pseudo forward pass initialization of network
model.eval()
B, N, F = 3, config.training_parameters.num_csts, config.training_parameters.num_features
csts = torch.randn(B, N, F).to(dtype=dtype_real)
mask = torch.ones(B, N, dtype=torch.bool)
x = torch.cat([csts, mask.unsqueeze(-1).to(csts.dtype)], dim=-1)
with torch.no_grad():
    model(x)

This is the model architecture. We should see the model FeatureClassifier, the submodules and for each layer, the quantizers and the pruning mask.

In [ ]:
print(model)

In [ ]:
trainer = L.Trainer(
    default_root_dir=save_path,
    max_epochs=max_epochs,
    accelerator="auto",
    devices="auto",
    precision="bf16-mixed",
    val_check_interval=1.0,
    callbacks=[
        ModelCheckpoint(dirpath=save_path, filename="best",
                        monitor="valid/total_loss", mode="min", save_top_k=1),
        ModelCheckpoint(dirpath=save_path, filename="last"),
        LearningRateMonitor(logging_interval="step"),
    ]
    
)

In [ ]:
model.train()
trainer.fit(model, datamodule=dataset)

In [ ]:
model.eval()
fig, ax = plot_remaining_weights(model)
plt.savefig(f"{save_path}/remaining_weights_pdp.png")
torch.save(model.state_dict(), f"{save_path}/state_manual_pdp.pth")
state_dict_export(model, f"{save_path}/model_state_dict_pdp.log") # this creates a huge text file where the state dict is logged for debugging purposes.



Now we use HGQ instead of "normal" quantization. HGQ (High-Granularity Quantization) uses differentiable quantization to find optimal number of bits.

In [ ]:
config = pdp_config() 

config.training_parameters.pretraining_epochs = 15 # training before start of pruning
config.training_parameters.epochs = 30 # training with pruning
config.training_parameters.fine_tuning_epochs = 10 # training after end of pruning, pruning mask is fixed here
max_epochs = config.training_parameters.pretraining_epochs + config.training_parameters.epochs + config.training_parameters.fine_tuning_epochs
# added manually to config for tuple input of model
config.training_parameters.num_csts = 40 # amount of constituents
config.training_parameters.num_features = 20 # amount of features

config.quantization_parameters.enable_quantization = True
config.quantization_parameters.granularity = "per_weight"
# granularity -> per tensor/channel/weight 
config.quantization_parameters.use_relu_multiplier = False
config.quantization_parameters.use_high_granularity_quantization = True
config.quantization_parameters.hgq_beta = 1e-12
config.quantization_parameters.hgq_gamma = 1e-6
config.quantization_parameters.overflow_mode_parameters = "SAT"
config.quantization_parameters.overflow_mode_data = "SAT"

config.quantization_parameters.default_data_keep_negatives = 0.
config.quantization_parameters.dynamic_data_quantization = True
config.quantization_parameters.default_data_integer_bits = 0.
config.quantization_parameters.default_data_fractional_bits = 8. # give all weights to fractional by default if dynamical data quantization

config.quantization_parameters.default_weight_keep_negatives = 1.
config.quantization_parameters.default_weight_integer_bits = 0.
config.quantization_parameters.default_weight_fractional_bits = 6. # give all weights to fractional by default if dynamical data quantization

config.pruning_parameters.enable_pruning = True
#config.pruning_parameters.max_pruning_pct = 1.00 # DST setting
#config.pruning_parameters.alpha = 1e-3 # DST setting
config.pruning_parameters.epsilon = 0.037
config.pruning_parameters.sparsity = 0.90
config.pruning_parameters.structured_pruning = False

config.training_parameters.num_csts = 40   
config.training_parameters.num_features = 20    

In [ ]:
n_classes = 4
d_model = 128
n_heads = 8 # does not change EBOPs!
num_layers = 4
dim_feedforward = 512
activation = 'relu' # originally gelu, but for compression, usually RELU is used because its easier
learning_rate = 1e-3
pooling = 'mean'
input_dim = config.training_parameters.num_features 

model = FeatureClassifier(
    config=config, 
    n_classes=n_classes, 
    d_model=d_model,
    n_heads=n_heads,
    num_layers=num_layers,
    dim_feedforward=dim_feedforward,
    activation=activation,
    learning_rate=learning_rate,
    pooling=pooling,
    # these are tunable hyperparameters as well, but they are also pruning/quantization-dependent and do not have that big influence compared to learning rate etc.
    scheduler=partial(warmup_cosine_scheduler, warmup_epochs=3, min_lr=1e-5),
    input_dim=input_dim
    )
add_compression_layers(model, config)

In [ ]:
# Pseudo forward pass initialization of network
model.eval()
B, N, F = 3, config.training_parameters.num_csts, config.training_parameters.num_features
csts = torch.randn(B, N, F).to(dtype=dtype_real)
mask = torch.ones(B, N, dtype=torch.bool)
x = torch.cat([csts, mask.unsqueeze(-1).to(csts.dtype)], dim=-1)
with torch.no_grad():
    model(x)

In [ ]:
model.train()
trainer.fit(model, datamodule=dataset)

In [ ]:
model.eval()
fig, ax = plot_remaining_weights(model)
plt.savefig(f"{save_path}/remaining_weights_pdp_hgq.png")
torch.save(model.state_dict(), f"{save_path}/state_manual_pdp_hgq.pth")
state_dict_export(model, f"{save_path}/model_state_dict_pdp_hgq.log") # this creates a huge text file where the state dict is logged for debugging purposes.

We repeat this with DST pruning, where we do not set an explicit sparsity, only the "strength" of pruning (ratio of pruning loss vs. cross-entropy loss) and the maximum pruning ratio (to prevent the whole layer getting pruned, which is okay in case of skip connections but would possibly lead to dead weights in other layeres). We import 'dst_config' instead of 'pdp_config'.

\alpha = 1e-3 is very aggressive, but this is okay in our network as it is overdimensioned either way.

In [ ]:
config = dst_config() 

config.training_parameters.pretraining_epochs = 0 # training before start of pruning
config.training_parameters.epochs = 30 # training with pruning
config.training_parameters.fine_tuning_epochs = 0 # training after end of pruning, pruning mask is fixed here
max_epochs = config.training_parameters.pretraining_epochs + config.training_parameters.epochs + config.training_parameters.fine_tuning_epochs
# added manually to config for tuple input of model
config.training_parameters.num_csts = 40 # amount of constituents
config.training_parameters.num_features = 20 # amount of features

config.quantization_parameters.enable_quantization = True
config.quantization_parameters.granularity = "per_weight"
# granularity -> per tensor/channel/weight 
config.quantization_parameters.use_relu_multiplier = False
config.quantization_parameters.use_high_granularity_quantization = False
config.quantization_parameters.hgq_beta = 1e-13
config.quantization_parameters.hgq_gamma = 1e-6
config.quantization_parameters.overflow_mode_parameters = "SAT"
config.quantization_parameters.overflow_mode_data = "SAT"

config.quantization_parameters.default_data_keep_negatives = 0.
config.quantization_parameters.dynamic_data_quantization = True
config.quantization_parameters.default_data_integer_bits = 0.
config.quantization_parameters.default_data_fractional_bits = 8. # give all weights to fractional by default if dynamical data quantization

config.quantization_parameters.default_weight_keep_negatives = 1.
config.quantization_parameters.default_weight_integer_bits = 0.
config.quantization_parameters.default_weight_fractional_bits = 6. # give all weights to fractional by default if dynamical data quantization

config.pruning_parameters.enable_pruning = True
config.pruning_parameters.max_pruning_pct = 1.00 # DST setting
config.pruning_parameters.alpha = 1e-3 # DST setting
#config.pruning_parameters.epsilon = 0.037
#config.pruning_parameters.sparsity = 0.90
#config.pruning_parameters.structured_pruning = False

config.training_parameters.num_csts = 40   
config.training_parameters.num_features = 20    

In [ ]:
n_classes = 4
d_model = 128
n_heads = 8 # does not change EBOPs!
num_layers = 4
dim_feedforward = 512
activation = 'relu' # originally gelu, but for compression, usually RELU is used because its easier
learning_rate = 1e-3
pooling = 'mean'
input_dim = config.training_parameters.num_features 

model = FeatureClassifier(
    config=config, 
    n_classes=n_classes, 
    d_model=d_model,
    n_heads=n_heads,
    num_layers=num_layers,
    dim_feedforward=dim_feedforward,
    activation=activation,
    learning_rate=learning_rate,
    pooling=pooling,
    # these are tunable hyperparameters as well, but they are also pruning/quantization-dependent and do not have that big influence compared to learning rate etc.
    scheduler=partial(warmup_cosine_scheduler, warmup_epochs=3, min_lr=1e-5),
    input_dim=input_dim
    )
add_compression_layers(model, config)

In [ ]:
# Pseudo forward pass initialization of network
model.eval()
B, N, F = 3, config.training_parameters.num_csts, config.training_parameters.num_features
csts = torch.randn(B, N, F).to(dtype=dtype_real)
mask = torch.ones(B, N, dtype=torch.bool)
x = torch.cat([csts, mask.unsqueeze(-1).to(csts.dtype)], dim=-1)
with torch.no_grad():
    model(x)

In [ ]:
model.train()
trainer.fit(model, datamodule=dataset)

In [ ]:
model.eval()
fig, ax = plot_remaining_weights(model)
plt.savefig(f"{save_path}/remaining_weights_dst.png")
torch.save(model.state_dict(), f"{save_path}/state_manual_dst.pth")
state_dict_export(model, f"{save_path}/model_state_dict_dst.log") # this creates a huge text file where the state dict is logged for debugging purposes.

The model is really overparametrized in its original (128, 512, 4)-shape. We will show this by training a (32, 32, 5) model to comparable accuracy (although without compression at first)

In [ ]:
config = dst_config() 

config.training_parameters.pretraining_epochs = 0 # training before start of pruning
config.training_parameters.epochs = 30 # training with pruning
config.training_parameters.fine_tuning_epochs = 0 # training after end of pruning, pruning mask is fixed here
max_epochs = config.training_parameters.pretraining_epochs + config.training_parameters.epochs + config.training_parameters.fine_tuning_epochs
# added manually to config for tuple input of model
config.training_parameters.num_csts = 40 # amount of constituents
config.training_parameters.num_features = 20 # amount of features

config.quantization_parameters.enable_quantization = False
config.quantization_parameters.granularity = "per_weight"
# granularity -> per tensor/channel/weight 
config.quantization_parameters.use_relu_multiplier = False
config.quantization_parameters.use_high_granularity_quantization = False
config.quantization_parameters.hgq_beta = 1e-13
config.quantization_parameters.hgq_gamma = 1e-6
config.quantization_parameters.overflow_mode_parameters = "SAT"
config.quantization_parameters.overflow_mode_data = "SAT"

config.quantization_parameters.default_data_keep_negatives = 0.
config.quantization_parameters.dynamic_data_quantization = True
config.quantization_parameters.default_data_integer_bits = 0.
config.quantization_parameters.default_data_fractional_bits = 8. # give all weights to fractional by default if dynamical data quantization

config.quantization_parameters.default_weight_keep_negatives = 1.
config.quantization_parameters.default_weight_integer_bits = 0.
config.quantization_parameters.default_weight_fractional_bits = 6. # give all weights to fractional by default if dynamical data quantization

config.pruning_parameters.enable_pruning = False
config.pruning_parameters.max_pruning_pct = 1.00 # DST setting
config.pruning_parameters.alpha = 1e-3 # DST setting
#config.pruning_parameters.epsilon = 0.037
#config.pruning_parameters.sparsity = 0.90
#config.pruning_parameters.structured_pruning = False

config.training_parameters.num_csts = 40   
config.training_parameters.num_features = 20    

In [ ]:
n_classes = 4
d_model = 32
n_heads = 8 # does not change EBOPs!
num_layers = 5
dim_feedforward = 32
activation = 'relu' # originally gelu, but for compression, usually RELU is used because its easier
learning_rate = 1e-3
pooling = 'mean'
input_dim = config.training_parameters.num_features 

model = FeatureClassifier(
    config=config, 
    n_classes=n_classes, 
    d_model=d_model,
    n_heads=n_heads,
    num_layers=num_layers,
    dim_feedforward=dim_feedforward,
    activation=activation,
    learning_rate=learning_rate,
    pooling=pooling,
    # these are tunable hyperparameters as well, but they are also pruning/quantization-dependent and do not have that big influence compared to learning rate etc.
    scheduler=partial(warmup_cosine_scheduler, warmup_epochs=3, min_lr=1e-5),
    input_dim=input_dim
    )
add_compression_layers(model, config)

In [ ]:
# Pseudo forward pass initialization of network
model.eval()
B, N, F = 3, config.training_parameters.num_csts, config.training_parameters.num_features
csts = torch.randn(B, N, F).to(dtype=dtype_real)
mask = torch.ones(B, N, dtype=torch.bool)
x = torch.cat([csts, mask.unsqueeze(-1).to(csts.dtype)], dim=-1)
with torch.no_grad():
    model(x)

In [ ]:
model.train()
trainer.fit(model, datamodule=dataset)

In [ ]:
model.eval()
torch.save(model.state_dict(), f"{save_path}/state_manual_small.pth")
state_dict_export(model, f"{save_path}/model_state_dict_small.log") # this creates a huge text file where the state dict is logged for debugging purposes.

In [ ]:
config = dst_config() 

config.training_parameters.pretraining_epochs = 0 # training before start of pruning
config.training_parameters.epochs = 30 # training with pruning
config.training_parameters.fine_tuning_epochs = 0 # training after end of pruning, pruning mask is fixed here
max_epochs = config.training_parameters.pretraining_epochs + config.training_parameters.epochs + config.training_parameters.fine_tuning_epochs
# added manually to config for tuple input of model
config.training_parameters.num_csts = 40 # amount of constituents
config.training_parameters.num_features = 20 # amount of features

config.quantization_parameters.enable_quantization = False
config.quantization_parameters.granularity = "per_weight"
# granularity -> per tensor/channel/weight 
config.quantization_parameters.use_relu_multiplier = False
config.quantization_parameters.use_high_granularity_quantization = False
config.quantization_parameters.hgq_beta = 1e-13
config.quantization_parameters.hgq_gamma = 1e-6
config.quantization_parameters.overflow_mode_parameters = "SAT"
config.quantization_parameters.overflow_mode_data = "SAT"

config.quantization_parameters.default_data_keep_negatives = 0.
config.quantization_parameters.dynamic_data_quantization = True
config.quantization_parameters.default_data_integer_bits = 0.
config.quantization_parameters.default_data_fractional_bits = 8. # give all weights to fractional by default if dynamical data quantization

config.quantization_parameters.default_weight_keep_negatives = 1.
config.quantization_parameters.default_weight_integer_bits = 0.
config.quantization_parameters.default_weight_fractional_bits = 6. # give all weights to fractional by default if dynamical data quantization

config.pruning_parameters.enable_pruning = False
config.pruning_parameters.max_pruning_pct = 1.00 # DST setting
config.pruning_parameters.alpha = 1e-3 # DST setting
#config.pruning_parameters.epsilon = 0.037
#config.pruning_parameters.sparsity = 0.90
#config.pruning_parameters.structured_pruning = False

config.training_parameters.num_csts = 40   
config.training_parameters.num_features = 20    

In [ ]:
n_classes = 4
d_model = 32
n_heads = 8 # does not change EBOPs!
num_layers = 5
dim_feedforward = 32
activation = 'relu' # originally gelu, but for compression, usually RELU is used because its easier
learning_rate = 1e-3
pooling = 'mean'
input_dim = config.training_parameters.num_features 

model = FeatureClassifier(
    config=config, 
    n_classes=n_classes, 
    d_model=d_model,
    n_heads=n_heads,
    num_layers=num_layers,
    dim_feedforward=dim_feedforward,
    activation=activation,
    learning_rate=learning_rate,
    pooling=pooling,
    # these are tunable hyperparameters as well, but they are also pruning/quantization-dependent and do not have that big influence compared to learning rate etc.
    scheduler=partial(warmup_cosine_scheduler, warmup_epochs=3, min_lr=1e-5),
    input_dim=input_dim
    )
add_compression_layers(model, config)

In [ ]:
# Pseudo forward pass initialization of network
model.eval()
B, N, F = 3, config.training_parameters.num_csts, config.training_parameters.num_features
csts = torch.randn(B, N, F).to(dtype=dtype_real)
mask = torch.ones(B, N, dtype=torch.bool)
x = torch.cat([csts, mask.unsqueeze(-1).to(csts.dtype)], dim=-1)
with torch.no_grad():
    model(x)

In [ ]:
model.train()
trainer.fit(model, datamodule=dataset)

In [ ]:
model.eval()
fig, ax = plot_remaining_weights(model)
plt.savefig(f"{save_path}/remaining_weights_small_compressed.png")
torch.save(model.state_dict(), f"{save_path}/state_manual_small_compressed.pth")
state_dict_export(model, f"{save_path}/model_state_dict_small_compressed.log") # this creates a huge text file where the state dict is logged for debugging purposes.

Now we try FITcompress (might take a bit longer - couple of hours).
FITcompress assigns each layer its individual bitwidth. At the end of the notebook, we see an example. It is noted that the attention layers require less bits than the other ones.

In [ ]:
config_fit = fitcompress_config() 

config_fit.training_parameters.pretraining_epochs = 20 # training before start of pruning
config_fit.training_parameters.epochs = 20 # training with pruning
config_fit.training_parameters.rounds = 2 # training with pruning
config_fit.training_parameters.fine_tuning_epochs = 10 # training after end of pruning, pruning mask is fixed here
max_epochs = config_fit.training_parameters.pretraining_epochs + config_fit.training_parameters.epochs + config_fit.training_parameters.fine_tuning_epochs
# added manually to config for tuple input of model
config_fit.training_parameters.num_csts = 40 # amount of constituents
config_fit.training_parameters.num_features = 20 # amount of features

config_fit.quantization_parameters.enable_quantization = True
config_fit.quantization_parameters.granularity = "per_weight"
# granularity -> per tensor/channel/weight 
config_fit.quantization_parameters.use_relu_multiplier = False
config_fit.quantization_parameters.use_high_granularity_quantization = False
config_fit.quantization_parameters.hgq_beta = 1e-13
config_fit.quantization_parameters.hgq_gamma = 1e-6
config_fit.quantization_parameters.overflow_mode_parameters = "SAT"
config_fit.quantization_parameters.overflow_mode_data = "SAT"

config_fit.quantization_parameters.default_data_keep_negatives = 0.
config_fit.quantization_parameters.dynamic_data_quantization = True
config_fit.quantization_parameters.default_data_integer_bits = 0.
config_fit.quantization_parameters.default_data_fractional_bits = 8. # give all weights to fractional by default if dynamical data quantization

config_fit.quantization_parameters.default_weight_keep_negatives = 1.
config_fit.quantization_parameters.default_weight_integer_bits = 0.
config_fit.quantization_parameters.default_weight_fractional_bits = 6. # give all weights to fractional by default if dynamical data quantization

config_fit.pruning_parameters.enable_pruning = True
#config_fit.pruning_parameters.max_pruning_pct = 1.00 # DST setting
#config_fit.pruning_parameters.alpha = 1e-3 # DST setting
config_fit.pruning_parameters.epsilon = 0.037
config_fit.pruning_parameters.sparsity = 0.90
config_fit.pruning_parameters.structured_pruning = False

config_fit.fitcompress_parameters.compression_goal = 0.002
config_fit.fitcompress_parameters.f_lambda = 1.
config_fit.fitcompress_parameters.quantization_schedule = [15., 11., 8., 7., 6., 5., 4., 3., 2.]
config_fit.fitcompress_parameters.optimize_pruning = True
config_fit.fitcompress_parameters.enable_fitcompress = True

config_fit.training_parameters.num_csts = 40   
config_fit.training_parameters.num_features = 20    

In [ ]:
model = FeatureClassifier(
    config=config_fit, 
    n_classes=n_classes, 
    d_model=d_model,
    n_heads=n_heads,
    num_layers=num_layers,
    dim_feedforward=dim_feedforward,
    activation=activation,
    learning_rate=learning_rate,
    pooling=pooling,
    # these are tunable hyperparameters as well, but they are also pruning/quantization-dependent and do not have that big influence compared to learning rate etc.
    scheduler=partial(warmup_cosine_scheduler, warmup_epochs=3, min_lr=1e-5),
    input_dim=input_dim
    )

In [ ]:
add_compression_layers(model, config)

To construct the compression layers, we make a random forward pass.

In [ ]:
# Pseudo forward pass initialization of network
model.eval()
B, N, F = 3, config.training_parameters.num_csts, config.training_parameters.num_features
csts = torch.randn(B, N, F).to(dtype=dtype_real)
mask = torch.ones(B, N, dtype=torch.bool)
x = torch.cat([csts, mask.unsqueeze(-1).to(csts.dtype)], dim=-1)
with torch.no_grad():
    model(x)

In [ ]:
print(f"Trainer initialized at {timestamp()}, starting fitting")

model.train()
trainer.fit(model, datamodule=dataset)

In [ ]:
print(f"Model fit finished at {timestamp()}. Now setting into evaluation mode")
model.eval()

fig, ax = plot_bit_allocation(model)
plt.savefig(f"{save_path}/bits_allocation.png", dpi=600)

torch.save(model.state_dict(), f"{save_path}/state_manual_fit.pth")

state_dict_export(model, f"{save_path}/model_state_dict_fit.log")

Experience with different pruning algorithms:
- DST: Works
- PDP: Sparsity goal can be predefined, but can be aggressive when structured pruning is active.
- Wanda: Does not work on our model (atm) because it requires a 2D input of the form (batch_size x features), while our input is 3D (batch_size x num_constituents x num_features)
- FIT: Takes a long time, gives to each layer individual quantizations, but does not get saved properly in state dict?

Quantizing this model (original size):
- Works until ca. 4-bit weight, 8-bit data

![Different quantization bitwidths](/shared/logs/quantize-grid-wd/output/heatmap_accuracy.png)

The tensorboard logs are stored in /shared/logs/*timestamp_path*. I added confusion matrices as well to the validation logs to see potential model collapse:

![Confusion matrix before compression](/shared/figs/before_pruning_confusion_matrix.png)

![Confusion matrix with PDP pruning, structured, 25% sparsity](/shared/figs/25pct_structured_pruning_confusion_matrix.png)

![Confusion matrix with PDP pruning, structured, 30% sparsity: One class collapses](/shared/figs/30pct_structured_pruning_confusion_matrix.png)

![Confusion matrix with DST pruning, unstructured, 99% sparsity: No collapses!](/shared/figs/dst_pruning_confusion_matrix_1pct.png)

![Confusion matrix with DST pruning, unstructured, 98% sparsity, quantization with 6-bit weight, 10-bit data: No collapses!](/shared/figs/dst_pruning_confusion_matrix_2pct-quant.png)

![Confusion matrix without pruning, quantization with 8-bit weight, 8-bit data: Collapse during training! (likely local minima)](/shared/figs/before_pruning_confusion_matrix_8bit_collapse.png)

![Confusion matrix without pruning, quantization with 24-bit weight, 24-bit data: Collapse during training! (likely local minima as this happens even with high bit quantization)](/shared/figs/before_pruning_confusion_matrix_24bit_collapse.png)

If we use FITcompress, we can set the compression ratio ourselves and the algorithm tries to find the optimal path to there. It looks like that:
Note that softmax is the only thing having  integer bits (to be expected for exp/inv)! Also, attention blocks really use few bits.

![Distribution integer/fractional bits with FITcompress](/shared/figs/bit_allocation.png)